In [16]:
# Import necessary libraries
import requests
import json
import pandas as pd
import altair as alt
import os

# Configure Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [40]:
# Define all 27 current EU member states
countries = {
    'AUT': 'Austria',
    'BEL': 'Belgium',
    'BGR': 'Bulgaria',
    'HRV': 'Croatia',
    'CYP': 'Cyprus',
    'CZE': 'Czech Republic',
    'DNK': 'Denmark',
    'EST': 'Estonia',
    'FIN': 'Finland',
    'FRA': 'France',
    'DEU': 'Germany',
    'GRC': 'Greece',
    'HUN': 'Hungary',
    'IRL': 'Ireland',
    'ITA': 'Italy',
    'LVA': 'Latvia',
    'LTU': 'Lithuania',
    'LUX': 'Luxembourg',
    'MLT': 'Malta',
    'NLD': 'Netherlands',
    'POL': 'Poland',
    'PRT': 'Portugal',
    'ROU': 'Romania',
    'SVK': 'Slovakia',
    'SVN': 'Slovenia',
    'ESP': 'Spain',
    'SWE': 'Sweden'
}

# World Bank API configuration
indicator = 'SE.XPD.TOTL.GD.ZS'  # Education expenditure as % of GDP
base_url = 'https://api.worldbank.org/v2/country'
date_range = '1995:2023'  # Time series range

# Create data folder if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Loop through countries and download data as JSON files
for country_code, country_name in countries.items():
    # Construct API URL
    url = f"{base_url}/{country_code}/indicator/{indicator}?date={date_range}&format=json&per_page=100"
    
    print(f"Downloading data for {country_name} ({country_code})...")
    
    # Make API request
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        
        # Save raw JSON file to data folder with new naming pattern
        country_name_formatted = country_name.replace(' ', '_').lower()
        filename = f"../data/eu_education_{country_name_formatted}_data.json"
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"  ✓ Saved to {filename}")
    else:
        print(f"  ✗ Error downloading data for {country_name}: {response.status_code}")

print(f"\nBatch download complete! {len(countries)} JSON files saved to data folder.")

  ✓ Saved to ../data/eu_education_austria_data.json
  ✓ Saved to ../data/eu_education_belgium_data.json
  ✓ Saved to ../data/eu_education_bulgaria_data.json
  ✓ Saved to ../data/eu_education_croatia_data.json
  ✓ Saved to ../data/eu_education_cyprus_data.json
  ✓ Saved to ../data/eu_education_czech_republic_data.json
  ✓ Saved to ../data/eu_education_denmark_data.json
  ✓ Saved to ../data/eu_education_estonia_data.json
  ✓ Saved to ../data/eu_education_finland_data.json
  ✓ Saved to ../data/eu_education_france_data.json
  ✓ Saved to ../data/eu_education_germany_data.json
  ✓ Saved to ../data/eu_education_greece_data.json
  ✓ Saved to ../data/eu_education_hungary_data.json
  ✓ Saved to ../data/eu_education_ireland_data.json
  ✓ Saved to ../data/eu_education_italy_data.json
  ✓ Saved to ../data/eu_education_latvia_data.json
  ✓ Saved to ../data/eu_education_lithuania_data.json
  ✓ Saved to ../data/eu_education_luxembourg_data.json
  ✓ Saved to ../data/eu_education_malta_data.json
  ✓ Sav

In [41]:
# Verify all JSON files exist
print("Verifying downloaded JSON files...")
for country_code, country_name in countries.items():
    country_name_formatted = country_name.replace(' ', '_').lower()
    filename = f"../data/eu_education_{country_name_formatted}_data.json"
    if os.path.exists(filename):
        print(f"✓ {filename}")
    else:
        print(f"✗ Missing: {filename}")

print(f"\nAll {len(countries)} raw JSON files ready for charts.")

Verifying downloaded JSON files...
✓ ../data/eu_education_austria_data.json
✓ ../data/eu_education_belgium_data.json
✓ ../data/eu_education_bulgaria_data.json
✓ ../data/eu_education_croatia_data.json
✓ ../data/eu_education_cyprus_data.json
✓ ../data/eu_education_czech_republic_data.json
✓ ../data/eu_education_denmark_data.json
✓ ../data/eu_education_estonia_data.json
✓ ../data/eu_education_finland_data.json
✓ ../data/eu_education_france_data.json
✓ ../data/eu_education_germany_data.json
✓ ../data/eu_education_greece_data.json
✓ ../data/eu_education_hungary_data.json
✓ ../data/eu_education_ireland_data.json
✓ ../data/eu_education_italy_data.json
✓ ../data/eu_education_latvia_data.json
✓ ../data/eu_education_lithuania_data.json
✓ ../data/eu_education_luxembourg_data.json
✓ ../data/eu_education_malta_data.json
✓ ../data/eu_education_netherlands_data.json
✓ ../data/eu_education_poland_data.json
✓ ../data/eu_education_portugal_data.json
✓ ../data/eu_education_romania_data.json
✓ ../data/eu_

In [55]:
# Create individual charts for each country using a loop
# Load data from raw JSON files and embed processed data in charts
charts = {}

# Function to load and process World Bank JSON
def process_wb_data(filename):
    with open(filename, 'r') as f:
        data = json.load(f)
    
    records = []
    if len(data) > 1 and data[1]:
        for item in data[1]:
            if item['value'] is not None:
                records.append({
                    'year': int(item['date']),
                    'education_gdp_pct': item['value']
                })
    return records

for country_code, country_name in sorted(countries.items()):
    # Load and process data from raw JSON file
    country_name_formatted = country_name.replace(' ', '_').lower()
    json_file = f"../data/eu_education_{country_name_formatted}_data.json"
    
    # Process the data
    processed_data = process_wb_data(json_file)
    
    # Create chart with embedded processed data
    chart = alt.Chart(alt.Data(values=processed_data)).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('year:Q', title='Year', axis=alt.Axis(grid=False)),
        y=alt.Y('education_gdp_pct:Q', title='Education Expenditure (% of GDP)', 
                scale=alt.Scale(zero=False), axis=alt.Axis(grid=False)),
        tooltip=[
            alt.Tooltip('year:Q', title='Year'),
            alt.Tooltip('education_gdp_pct:Q', format='.2f', title='% of GDP')
        ]
    ).properties(
        width=300,
        height=200,
        title=f'{country_name}: Education Expenditure as % of GDP'
    ).interactive()
    
    charts[country_code] = chart
    print(f"✓ Created chart for {country_name} with {len(processed_data)} data points")

print(f"\nTotal charts created: {len(charts)}")

✓ Created chart for Austria with 27 data points
✓ Created chart for Belgium with 18 data points
✓ Created chart for Bulgaria with 22 data points
✓ Created chart for Cyprus with 23 data points
✓ Created chart for Czech Republic with 27 data points
✓ Created chart for Germany with 17 data points
✓ Created chart for Denmark with 26 data points
✓ Created chart for Spain with 27 data points
✓ Created chart for Estonia with 27 data points
✓ Created chart for Finland with 26 data points
✓ Created chart for France with 9 data points
✓ Created chart for Greece with 13 data points
✓ Created chart for Croatia with 14 data points
✓ Created chart for Hungary with 27 data points
✓ Created chart for Ireland with 26 data points
✓ Created chart for Italy with 27 data points
✓ Created chart for Lithuania with 26 data points
✓ Created chart for Luxembourg with 12 data points
✓ Created chart for Latvia with 26 data points
✓ Created chart for Malta with 17 data points
✓ Created chart for Netherlands with 2

In [56]:
# Create graphs folder if it doesn't exist
os.makedirs('../graphs', exist_ok=True)

# Save all charts as JSON specification files
for country_code, chart in charts.items():
    country_name = countries[country_code].replace(' ', '_').lower()
    filename = f"../graphs/eu_education_{country_name}.json"
    chart.save(filename)
    print(f"Saved {filename}")

print(f"\n✓ All {len(charts)} charts saved to graphs folder!")

Saved ../graphs/eu_education_austria.json
Saved ../graphs/eu_education_belgium.json
Saved ../graphs/eu_education_bulgaria.json
Saved ../graphs/eu_education_cyprus.json
Saved ../graphs/eu_education_czech_republic.json
Saved ../graphs/eu_education_germany.json
Saved ../graphs/eu_education_denmark.json
Saved ../graphs/eu_education_spain.json
Saved ../graphs/eu_education_estonia.json
Saved ../graphs/eu_education_finland.json
Saved ../graphs/eu_education_france.json
Saved ../graphs/eu_education_greece.json
Saved ../graphs/eu_education_croatia.json
Saved ../graphs/eu_education_hungary.json
Saved ../graphs/eu_education_ireland.json
Saved ../graphs/eu_education_italy.json
Saved ../graphs/eu_education_lithuania.json
Saved ../graphs/eu_education_luxembourg.json
Saved ../graphs/eu_education_latvia.json
Saved ../graphs/eu_education_malta.json
Saved ../graphs/eu_education_netherlands.json
Saved ../graphs/eu_education_poland.json
Saved ../graphs/eu_education_portugal.json
Saved ../graphs/eu_educatio